# fy-m1 01 — QLoRA training (T4)
Set `MODEL_ID` (the lead's pick) and `RUN_ID`, then **Runtime → Run all**. Trains a LoRA adapter on the private
`Fan1000-Prog/fyM1-chats` data (every `*.jsonl`, rows `{"id", "messages"}`, plus `system_prompt.txt`) at a pinned
dataset revision. Everything of a run lives under `fyM1-ckpt/<RUN_ID>/`: each `checkpoint-<step>/` carries
`run_config.json` (refused on resume if the notebook's config differs), `session_progress.json` (wall time and
session count so far) and `data_stats.json`. Rerunning after a disconnect resumes the same run; a new lr or data
means a new `RUN_ID`. Training stops when eval loss rises 2 evals in a row (or is NaN) and writes
`<RUN_ID>/stopped.json`; reruns then skip straight to the final cell. The best checkpoint becomes `<RUN_ID>/final/`
and `final/`; the log goes to `<RUN_ID>/train_log.json` and `results/train_log.json`.
Runs on Colab or Kaggle (T4). Needs the `HF_TOKEN` secret (Colab: Secrets pane with notebook access; Kaggle: Add-ons → Secrets) and code synced from clean origin/main (`scripts/sync_code_to_hub.py`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U unsloth unsloth_zoo

In [ ]:
MODEL_ID = None  # set by the lead's model pick, e.g. "org/model"
RUN_ID = None    # e.g. "r1-lr2e-4"; one folder per run in fyM1-ckpt
LR, LR_SCHEDULER, WARMUP, EPOCHS, MAX_LEN = 2e-4, "cosine", 10, 2, 2048
BATCH, GRAD_ACCUM, SAVE_EVERY = 2, 4, 100
LORA_R, LORA_ALPHA = 16, 16
EVAL_SHARE, SEED = 0.02, 42
CKPT_REPO, DATA_REPO = "Fan1000-Prog/fyM1-ckpt", "Fan1000-Prog/fyM1-chats"
assert MODEL_ID and RUN_ID, "set MODEL_ID and RUN_ID"

In [ ]:
# Runtime bootstrap (Colab or Kaggle). The HF token is needed before fym1 can be downloaded, so this cell
# carries a verbatim copy of fym1.runtime.hf_token and its helpers (checked by tests/test_notebook_runtime.py).
import os
import tempfile

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Kaggle's T4 runtime may preset "0,1"; always use one GPU
CODE_DIR = tempfile.mkdtemp(prefix="fym1code-")  # fresh each run: nothing stale on sys.path

MISSING_TOKEN = ("no HF_TOKEN found: on Kaggle attach it under Add-ons → Secrets; on Colab add it in the "
                 "Secrets pane with notebook access; elsewhere set the HF_TOKEN environment variable")


def on_kaggle() -> bool:
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        return True
    try:
        import kaggle_secrets  # noqa: F401  (only installed in Kaggle kernels)
    except ImportError:
        return False
    return True


def _secret(get) -> str | None:
    try:
        return get("HF_TOKEN")
    except Exception:  # missing or not attached: the caller raises one clear error; never echo details
        return None


def hf_token() -> str:
    """HF token from Kaggle secrets, else Colab secrets, else the HF_TOKEN env var. Never print it."""
    if on_kaggle():
        from kaggle_secrets import UserSecretsClient

        token = _secret(UserSecretsClient().get_secret)
    else:
        try:
            from google.colab import userdata
        except ImportError:
            token = os.environ.get("HF_TOKEN")
        else:
            token = _secret(userdata.get)
    if not token:
        raise RuntimeError(MISSING_TOKEN)
    return token

In [ ]:
import sys, json, time, random, shutil
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download, snapshot_download, list_repo_files

SESSION_START = time.time()
os.environ["HF_TOKEN"] = hf_token()  # never print it
api = HfApi()
code_dir = snapshot_download(CKPT_REPO, allow_patterns=["code/**"], local_dir=CODE_DIR)
sys.path.insert(0, f"{code_dir}/code/src")
CODE_SHA = open(f"{code_dir}/code/SHA").read().strip()
from fym1.train.resume import latest_checkpoint
from fym1.train.chat_format import CHAT_TEMPLATE_KWARGS, training_text, turn_markers
from fym1.train.supervise import best_eval, eval_losses, should_stop
from fym1.train.run_state import config_mismatch, session_progress
from fym1.runtime import output_dir, scratch_dir

OUTPUT, SCRATCH = output_dir(), scratch_dir()  # small wanted results vs big throwaway files
OUT = f"{SCRATCH}/out"  # checkpoints (pushed to the Hub, never kept as Kaggle output)

# Resume state for this run, read before anything trains.
repo_files = list_repo_files(CKPT_REPO)
resume = latest_checkpoint(CKPT_REPO, repo_files, run=RUN_ID)
STOPPED = f"{RUN_ID}/stopped.json" in repo_files
resumed_progress, saved_config = None, None
if resume:
    snapshot_download(CKPT_REPO, allow_patterns=[f"{resume}/*"], local_dir=OUT)
    saved_config = json.load(open(f"{OUT}/{resume}/run_config.json"))
    resumed_progress = json.load(open(f"{OUT}/{resume}/session_progress.json"))
DATA_REVISION = saved_config["data_revision"] if saved_config else api.dataset_info(DATA_REPO).sha
RUN_CONFIG = {"run_id": RUN_ID, "model": MODEL_ID, "lr": LR, "lr_scheduler": LR_SCHEDULER, "warmup_steps": WARMUP,
              "epochs": EPOCHS, "max_seq_len": MAX_LEN, "batch": BATCH, "grad_accum": GRAD_ACCUM,
              "save_every": SAVE_EVERY, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "eval_share": EVAL_SHARE,
              "seed": SEED, "data_revision": DATA_REVISION, "chat_template_kwargs": CHAT_TEMPLATE_KWARGS,
              "code_sha": CODE_SHA}  # a resume on other code is refused: no code sync to the Hub during a run
if saved_config:
    mismatch = config_mismatch(saved_config, RUN_CONFIG)
    assert not mismatch, f"{resume} belongs to a different run config: {mismatch}; use a new RUN_ID"
print("RUN", RUN_ID, "| resume from", resume or "scratch", "| stopped earlier:", STOPPED, "| code", CODE_SHA)

In [ ]:
import torch

if not STOPPED:
    from unsloth import FastModel

    model, processor = FastModel.from_pretrained(MODEL_ID, max_seq_length=MAX_LEN, load_in_4bit=True,
                                                 full_finetuning=False, use_exact_model_name=True)
    tok = getattr(processor, "tokenizer", processor)
    lora = dict(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none", random_state=SEED,
                use_gradient_checkpointing="unsloth")
    if hasattr(model.config, "vision_config"):
        model = FastModel.get_peft_model(model, finetune_vision_layers=False, finetune_language_layers=True,
                                         finetune_attention_modules=True, finetune_mlp_modules=True, **lora)
    else:
        model = FastModel.get_peft_model(model, target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                                                "gate_proj", "up_proj", "down_proj"], **lora)
    compute_dtype = next(p.dtype for p in model.parameters() if p.is_floating_point())
    print("model dtype", compute_dtype, "| vram GB", round(torch.cuda.memory_allocated() / 2**30, 2))

In [ ]:
if not STOPPED:
    from datasets import Dataset

    def render(messages, add_generation_prompt=False):
        return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt,
                                       **CHAT_TEMPLATE_KWARGS)

    files = [f for f in list_repo_files(DATA_REPO, repo_type="dataset", revision=DATA_REVISION) if f.endswith(".jsonl")]
    system_prompt = open(hf_hub_download(DATA_REPO, "system_prompt.txt", repo_type="dataset",
                                         revision=DATA_REVISION)).read().strip()
    rows = []
    for f in files:
        path = hf_hub_download(DATA_REPO, f, repo_type="dataset", revision=DATA_REVISION)
        rows += [json.loads(l) for l in open(path) if l.strip()]
    random.Random(SEED).shuffle(rows)
    texts = [training_text(render, [{"role": "system", "content": system_prompt}] + r["messages"]) for r in rows]
    n_eval = max(1, round(len(texts) * EVAL_SHARE))
    train_ds, eval_ds = Dataset.from_dict({"text": texts[n_eval:]}), Dataset.from_dict({"text": texts[:n_eval]})
    lengths = [len(tok(t, add_special_tokens=False)["input_ids"]) for t in texts]
    DATA_STATS = {"files": files, "revision": DATA_REVISION, "examples": len(texts), "train": len(train_ds),
                  "eval": n_eval, "tokens": sum(lengths), "over_max_len": sum(l > MAX_LEN for l in lengths)}
    print(DATA_STATS)

In [ ]:
if not STOPPED:
    from trl import SFTTrainer, SFTConfig
    from transformers import TrainerCallback
    from unsloth.chat_templates import train_on_responses_only

    class PushAndSupervise(TrainerCallback):
        """Push every checkpoint (with run config, progress, data stats) to the Hub in one commit; stop when
        eval loss rises 2 evals in a row or is NaN, and put a stopped.json marker in the stop-step commit."""

        stop_marker = None

        def on_save(self, args, state, control, **kw):
            name = f"checkpoint-{state.global_step}"
            folder = f"{args.output_dir}/{name}"
            extras = {"run_config.json": RUN_CONFIG, "data_stats.json": DATA_STATS,
                      "session_progress.json": session_progress(resumed_progress, time.time() - SESSION_START)}
            for fname, payload in extras.items():
                with open(f"{folder}/{fname}", "w") as f:
                    json.dump(payload, f, indent=2)
            ops = [CommitOperationAdd(path_in_repo=f"{RUN_ID}/{name}/{os.path.relpath(path, folder)}",
                                      path_or_fileobj=path)
                   for root, _, fnames in os.walk(folder) for path in (os.path.join(root, f) for f in fnames)]
            if self.stop_marker:
                ops.append(CommitOperationAdd(path_in_repo=f"{RUN_ID}/stopped.json",
                                              path_or_fileobj=json.dumps(self.stop_marker, indent=2).encode()))
            api.create_commit(repo_id=CKPT_REPO, operations=ops, commit_message=f"train {RUN_ID}: {name}")

        def on_evaluate(self, args, state, control, **kw):
            if should_stop([l for _, l in eval_losses(state.log_history)]):
                print("STOP: eval loss rose 2 evals in a row (or NaN)")
                control.should_training_stop = True
                control.should_save = True  # the stop-step checkpoint carries the marker (on_save)
                self.stop_marker = {"step": state.global_step, "reason": "eval loss rose 2 evals in a row or NaN",
                                    "eval_curve": eval_losses(state.log_history)}

    tok.padding_side = "right"
    trainer = SFTTrainer(
        model=model, processing_class=tok, train_dataset=train_ds, eval_dataset=eval_ds,
        callbacks=[PushAndSupervise()],
        args=SFTConfig(output_dir=f"{OUT}/{RUN_ID}", per_device_train_batch_size=BATCH,
                       per_device_eval_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
                       num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type=LR_SCHEDULER,
                       warmup_steps=WARMUP, logging_steps=10, optim="adamw_8bit",
                       fp16=compute_dtype == torch.float16, bf16=False, max_length=MAX_LEN,
                       dataset_text_field="text", packing=False, eval_strategy="steps", eval_steps=SAVE_EVERY,
                       save_strategy="steps", save_steps=SAVE_EVERY, save_total_limit=2, report_to="none",
                       seed=SEED))
    instruction_part, response_part = turn_markers(render)
    trainer = train_on_responses_only(trainer, instruction_part=instruction_part, response_part=response_part)
    trainer.train(resume_from_checkpoint=f"{OUT}/{resume}" if resume else None)
    trainer.save_model(f"{OUT}/{RUN_ID}/last")  # final weights, also when stopped early; not resumable
    api.upload_folder(folder_path=f"{OUT}/{RUN_ID}/last", path_in_repo=f"{RUN_ID}/last", repo_id=CKPT_REPO,
                      commit_message=f"train {RUN_ID}: last")
    print("TRAIN_DONE step", trainer.state.global_step)

In [ ]:
if STOPPED:  # trained in an earlier session: read its state from the newest checkpoint
    history = json.load(open(f"{OUT}/{resume}/trainer_state.json"))["log_history"]
    DATA_STATS = json.load(open(f"{OUT}/{resume}/data_stats.json"))
    progress, steps = resumed_progress, int(resume.rsplit("-", 1)[1])
else:
    history = trainer.state.log_history
    progress, steps = session_progress(resumed_progress, time.time() - SESSION_START), trainer.state.global_step
best = best_eval(history)
final_src = f"{RUN_ID}/checkpoint-{best[0]}" if best else f"{RUN_ID}/last"
local = snapshot_download(CKPT_REPO, allow_patterns=[f"{final_src}/*"], local_dir=f"{SCRATCH}/final_src")
adapter_files = [f for f in os.listdir(f"{local}/{final_src}") if f.startswith("adapter_") or f.startswith("tokenizer")]
assert any(f.startswith("adapter_") for f in adapter_files), f"no adapter files in {final_src}"
final_dir = f"{SCRATCH}/final"
shutil.rmtree(final_dir, ignore_errors=True)
os.makedirs(final_dir)
for f in adapter_files:
    shutil.copy(f"{local}/{final_src}/{f}", f"{final_dir}/{f}")
for dest in (f"{RUN_ID}/final", "final"):
    api.upload_folder(folder_path=final_dir, path_in_repo=dest, repo_id=CKPT_REPO,
                      commit_message=f"{dest} adapter = {final_src}")

losses = [h["loss"] for h in history if "loss" in h]
log = {
    "run_id": RUN_ID, "model": MODEL_ID, "code_sha": CODE_SHA, "gpu": torch.cuda.get_device_name(0),
    "steps": steps, "epochs": EPOCHS, "wall_time_s": progress["wall_time_s"], "sessions": progress["sessions"],
    "final_train_loss": losses[-1] if losses else None, "best_eval_loss": best[1] if best else None,
    "best_step": best[0] if best else None, "final_adapter": final_src,
    "stopped_early": f"{RUN_ID}/stopped.json" in list_repo_files(CKPT_REPO),
    "data": DATA_STATS,
    "config": {"lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lr": LR, "lr_scheduler": LR_SCHEDULER,
               "warmup_steps": WARMUP, "max_seq_len": MAX_LEN, "effective_batch": BATCH * GRAD_ACCUM,
               "data_revision": DATA_REVISION, "chat_template_kwargs": CHAT_TEMPLATE_KWARGS},
    "loss_curve": [{"step": h["step"], "loss": h["loss"]} for h in history if "loss" in h],
    "eval_curve": [{"step": s, "eval_loss": l} for s, l in eval_losses(history)],
}
body = json.dumps(log, indent=2).encode()
for dest in (f"{RUN_ID}/train_log.json", "results/train_log.json"):
    api.upload_file(path_or_fileobj=body, path_in_repo=dest, repo_id=CKPT_REPO, commit_message=f"train log {RUN_ID}")
with open(f"{OUTPUT}/train_log.json", "wb") as f:  # also a small run output (Kaggle: kernels output)
    f.write(body)
print("TRAIN_LOG", json.dumps({k: v for k, v in log.items() if not k.endswith("_curve")}))